In [1]:

"""
Chimanimani InSAR — upload correct files to Google Drive with space
monitoring and resumability.

Recomputes the tightest-25 pair list (track 72, AOI-verified, 40m
resolution), matches it against succeeded HyP3 jobs, and uploads each to
a dedicated Drive subfolder, renamed to its HyP3 job ID. Progress is
checked by reading that folder's existing filenames directly, and
uploads stop once free space drops below MIN_FREE_GB.
"""

import subprocess
import sys


def _ensure_installed(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


for _pkg in ("asf_search", "hyp3_sdk", "shapely", "geopandas"):
    _ensure_installed(_pkg)

import shutil
import tempfile
import zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path

import asf_search as asf
import geopandas as gpd
import hyp3_sdk as sdk
from google.colab import drive
from google.colab import files as colab_files
from shapely import wkt as shapely_wkt
from shapely.geometry import shape as shapely_shape

DRIVE_FOLDER_NAME = "Chimanimani_InSAR_Output"
MIN_FREE_GB = 1.0


def parse_time(iso_str):
    return datetime.fromisoformat(iso_str.replace("Z", "+00:00"))


def free_space_gb(path):
    return shutil.disk_usage(path).free / (1024 ** 3)


# ---------------------------------------------------------------------------
# Mount Drive under whichever account this session is signed into, and
# use a dedicated subfolder rather than Drive's root.
# ---------------------------------------------------------------------------
drive.mount("/content/drive")
drive_folder = Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME
drive_folder.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# AOI, from the team's shapefile
# ---------------------------------------------------------------------------
print("Upload your AOI shapefile as a single .zip:")
uploaded = colab_files.upload()
shapefile_zip_path = next(iter(uploaded))

extract_dir = tempfile.mkdtemp()
with zipfile.ZipFile(shapefile_zip_path) as z:
    z.extractall(extract_dir)
gdf = gpd.read_file(list(Path(extract_dir).glob("*.shp"))[0])
if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
merged_geom = gdf.union_all() if hasattr(gdf, "union_all") else gdf.unary_union
aoi_wkt = merged_geom.wkt
aoi_polygon = shapely_wkt.loads(aoi_wkt)


def aoi_overlap_fraction(scene):
    return shapely_shape(scene.geometry).intersection(aoi_polygon).area / aoi_polygon.area


# ---------------------------------------------------------------------------
# Recompute the validated consistent acquisition set on track 72
# ---------------------------------------------------------------------------
CONFIRMED_TRACK = 72  # verified: 99% AOI overlap

results = asf.geo_search(
    platform=[asf.PLATFORM.SENTINEL1],
    intersectsWith=aoi_wkt,
    processingLevel=asf.PRODUCT_TYPE.SLC,
    beamMode="IW",
    polarization="VV+VH",
    start=datetime(2025, 9, 17),
    end=datetime(2026, 9, 17),
)
results = [r for r in results if r.properties["platform"] in ("Sentinel-1C", "Sentinel-1D")]
same_track = [r for r in results if r.properties["pathNumber"] == CONFIRMED_TRACK]

combo_counts = Counter(
    (r.properties["beamModeType"], r.properties["polarization"], r.properties["flightDirection"])
    for r in same_track
)
dominant_combo = combo_counts.most_common(1)[0][0]
same_track = [
    r for r in same_track
    if (r.properties["beamModeType"], r.properties["polarization"], r.properties["flightDirection"]) == dominant_combo
]
same_track.sort(key=lambda r: parse_time(r.properties["startTime"]))

# ---------------------------------------------------------------------------
# Recompute the definitive tightest-25 pair list
# ---------------------------------------------------------------------------
reference = same_track[0]
same_track_scene_names = {r.properties["sceneName"] for r in same_track}
stack = [s for s in reference.stack() if s.properties["sceneName"] in same_track_scene_names]

TEMP_BASELINE_MAX = 48
PERP_BASELINE_MAX = 200
MAX_PAIRS = 25

candidates = []
for i, ref in enumerate(stack):
    for sec in stack[i + 1:]:
        dt = abs((parse_time(sec.properties["startTime"]) - parse_time(ref.properties["startTime"])).days)
        ref_b = ref.properties.get("perpendicularBaseline") or 0.0
        sec_b = sec.properties.get("perpendicularBaseline") or 0.0
        db = abs(sec_b - ref_b)
        if dt <= TEMP_BASELINE_MAX and db <= PERP_BASELINE_MAX:
            closeness = (dt / TEMP_BASELINE_MAX) + (db / PERP_BASELINE_MAX)
            candidates.append((closeness, ref.properties["sceneName"], sec.properties["sceneName"]))
candidates.sort(key=lambda c: c[0])
target_pairs = {frozenset((r, s)) for _, r, s in candidates[:MAX_PAIRS]}

# ---------------------------------------------------------------------------
# Match against succeeded HyP3 jobs
# ---------------------------------------------------------------------------
hyp3 = sdk.HyP3(prompt=True)
print(f"Remaining HyP3 credits: {hyp3.check_credits()}")

all_jobs = hyp3.find_jobs(name="chimanimani-batch", status_code="SUCCEEDED")
matched_jobs = [
    job for job in all_jobs
    if len(job.job_parameters.get("granules", [])) == 2
    and frozenset(job.job_parameters["granules"]) in target_pairs
]
print(f"{len(matched_jobs)} of {len(target_pairs)} correct pairs are succeeded and ready.")

# ---------------------------------------------------------------------------
# Upload to Drive, renamed by job ID; skip files already present
# ---------------------------------------------------------------------------
already_done = {f.stem for f in drive_folder.glob("*.zip")}
remaining_jobs = [job for job in matched_jobs if job.job_id not in already_done]
print(f"{len(already_done)} already in this account's folder, {len(remaining_jobs)} left to upload here.")

for i, job in enumerate(remaining_jobs, 1):
    free_gb = free_space_gb(drive_folder)
    if free_gb < MIN_FREE_GB:
        print(f"\nOnly {free_gb:.2f} GB free in this account — stopping here.")
        print(f"{len(remaining_jobs) - i + 1} pair(s) still need uploading.")
        break

    local_dir = tempfile.mkdtemp()
    downloaded = job.download_files(location=local_dir)
    if downloaded:
        shutil.move(str(downloaded[0]), str(drive_folder / f"{job.job_id}.zip"))
    print(f"[{i}/{len(remaining_jobs)}] uploaded (job {job.job_id}), {free_space_gb(drive_folder):.2f} GB free")
else:
    print(f"\nAll {len(matched_jobs)} correct files are now in this account's Drive folder.")

Mounted at /content/drive
Upload your AOI shapefile as a single .zip:


Saving Chimanimani_AOI_v2.zip to Chimanimani_AOI_v2.zip


["'type': 'REVERSE': 'report': Reversed polygon winding order"]
/usr/local/lib/python3.13/dist-packages/hyp3_sdk/hyp3.py:55: UserWarning: Passing `prompt=True` is deprecated. Please use either `prompt="password"` or `prompt="token"`
  warnings.warn(


Remaining HyP3 credits: 6995
25 of 25 correct pairs are succeeded and ready.
0 already in this account's folder, 25 left to upload here.


S1CD_20260522T162307_20260604T162318_VVP013_INT40_G_ueF_26CB.zip:   0%|          | 0/352347470 [00:00<?, ?it/s…

...

[10/25] uploaded (job e1c7919e-c36e-4120-acfe-30c5c034134b), 2.13 GB free


S1CC_20251229T162305_20260110T162305_VVP012_INT40_G_ueF_0068.zip:   0%|          | 0/354190437 [00:00<?, ?it/s…

[11/25] uploaded (job 9946dba7-8f2a-409b-8449-ddc5f1994d45), 2.13 GB free


S1CC_20260716T162307_20260728T162308_VVP012_INT40_G_ueF_4262.zip:   0%|          | 0/348211812 [00:00<?, ?it/s…

[12/25] uploaded (job 2c491011-d409-4565-a17f-0bcf15947828), 0.16 GB free

Only 0.16 GB free in this account — stopping here.
13 pair(s) still need uploading.


In [ ]:


import json
import time
from google.colab import _message, files

notebook_json = None
for attempt in range(3):
    response = _message.blocking_request("get_ipynb", timeout_sec=30)
    if response is not None:
        notebook_json = response["ipynb"]
        break
    time.sleep(2)

if notebook_json is not None:
    with open("SBAS insar.ipynb", "w") as f:
        json.dump(notebook_json, f)
    files.download("SBAS insar.ipynb")
else:
    print("Notebook API unavailable — use File > Download > Download .ipynb instead.")